# Conversational Information Gain (CIG)

`CIG` uses an LLM to rate how much new information each utterance adds to the conversation so far, from 1 (no gain) to 4 (insightful). This demo runs it on a small sample corpus of AI-mediated conversations and compares mediator turns with participant turns.

**Requirements**

- ConvoKit with its GenAI dependencies: `pip install "convokit[genai]"` (or, from a clone of the repository, `pip install -e ".[genai]"`).
- An OpenAI or Google Gemini API key. CIG makes one LLM call per utterance, a few hundred calls for the sample corpus, which are billed to your account.

The notebook loads the sample corpus from `sample_corpus/`, which sits next to this notebook in `convokit/ai_evaluation/examples/`. Open the notebook from that folder (or change the path in `Corpus.load`).

## Setup: choose your LLM

Set `PROVIDER` to the provider you have an API key for, and optionally change the model in `MODELS`:

- `"gemini"`: Google Gemini ([get a key](https://aistudio.google.com/apikey))
- `"gpt"`: OpenAI ([get a key](https://platform.openai.com/api-keys))

ConvoKit looks for the key in this order:

1. the `GEMINI_API_KEY` / `GPT_API_KEY` environment variable (set it before starting Jupyter, or paste the key into `API_KEY` below; don't commit a notebook with a key in it);
2. `~/.convokit/config.yml`. To save a key there once, run:
   ```python
   from convokit.genai import GenAIConfigManager
   GenAIConfigManager().set_api_key("gemini", "YOUR_KEY")  # or "gpt"
   ```

Gemini can also run through Vertex AI instead of an API key: run `gcloud auth application-default login`, then `GenAIConfigManager().set_google_cloud_config("your-project-id", "us-central1")`.

In [1]:
import os

PROVIDER = "gemini"  # "gemini" or "gpt"
MODELS = {
    "gpt": "gpt-5",
    "gemini": "gemini-2.5-flash",
}
API_KEY = ""  # optional; leave empty to use the environment variable or ~/.convokit/config.yml

if API_KEY:
    os.environ[f"{PROVIDER.upper()}_API_KEY"] = API_KEY
MODEL = MODELS[PROVIDER]
print("Using", MODEL)

Using gemini-2.5-flash


In [2]:
from convokit import Corpus
from convokit.ai_evaluation import CIG

corpus = Corpus.load("sample_corpus")
corpus.print_summary_stats()

Number of Speakers: 36
Number of Utterances: 220
Number of Conversations: 12
Number of AI Speakers: 36
Number of PrivateAssistants: 0
Number of Supports: 0


In [3]:
cig = CIG(model=MODEL)
cig.transform(corpus)

10 / 12


In [4]:
# per utterance: 1 (no gain) .. 4 (insightful); None = opening context or too short to rate
convo = next(c for c in corpus.iter_conversations() if c.meta["mediator"] == "constructive")
for utt in convo.get_chronological_utterance_list():
    print(utt.meta["cig"], convo.alias.get(utt.speaker.id, utt.speaker.id), ":", utt.text[:90])

None Mediator ([default] constructive mediator) : Welcome Goose and Bear. We are here to discuss the statement: 'I support local governments
None Goose : Data centers drive economic growth, create high-tech jobs, and provide essential infrastru
None Bear : Resource strain threatens local water and power stability. These long-term environmental c
3 Goose : Innovation leads to more efficient cooling and power usage, mitigating those environmental
2 Mediator ([default] constructive mediator) : Goose suggests that technological innovation in efficiency can mitigate environmental impa
2 Bear : Efficiency helps, but absolute demand still strains finite local resources. Innovation sho
3 Goose : Data centers often fund renewable energy projects, upgrading local grids and improving lon
3 Bear : Renewable offsets don't fix immediate local shortages. Communities shouldn't risk their ba
3 Mediator ([default] constructive mediator) : We have a clear tension between immediate local utility risks and

In [5]:
# mean per conversation, mediator turns vs participant turns; then per mediator type
summary = cig.summarize(corpus)
summary["mediator"] = summary.conversation_id.map(lambda c: corpus.get_conversation(c).meta["mediator"])
summary.groupby(["mediator", "is_mediator"]).cig_mean.mean()

mediator      is_mediator
constructive  False          3.214598
              True           2.625000
greeting      False          3.099650
              True           1.000000
paraphrase    False          2.900000
              True           1.062500
Name: cig_mean, dtype: float64